# Quick Start: Iterating Over GIFT-Eval Splits 

This notebook demonstrates how to iterate over GIFT-Eval's training, validation, and test splits. There's two things we need to do before getting started: 

1. Download the GIFT-Eval benchmark from Hugging Face [here](https://huggingface.co/datasets/Salesforce/GiftEval).
2. Set the `GIFT-EVAL` environment variable as the local path to the datasets downloaded from Hugging Face using ```echo "GIFT_EVAL=PATH_TO_SAVE" >> .env```

## Sanity Check

First, let's check the names of all the datasets in the `gift-eval` directory.

In [1]:
import os

from pathlib import Path
from dotenv import load_dotenv

# Readthe .env file and load environment variables
load_dotenv()

# Get the GIFT_EVAL path from environment variables
gift_eval_path = os.getenv("GIFT_EVAL")

# Convert GIFT_EVAL path to a Path object for easier filesystem operations
gift_eval_path = Path(gift_eval_path)

# Get all subdirectories (dataset names) in the GIFT_EVAL path
dataset_names = []

# Get all the directiories in the GIFT_EVAL path
directories = gift_eval_path.iterdir()

for directory in directories:
    # Ignore directories that start with "." or paths that aren't directories
    is_hidden_directory = directory.name.startswith(".")
    is_not_directory = not directory.is_dir()

    if is_hidden_directory or is_not_directory:
        continue

    directory_name = directory.name

    # Check if there a subdirectories containing data of different frequencies
    frequency_directories = [d for d in directory.iterdir() if d.is_dir()]

    # If there aren't any subdirectories, save the directory's name
    if not frequency_directories:
        dataset_names.append(directory_name)
        continue

    # For each subdirectory, save the subdirectory and directory's names
    for frequency_directory in frequency_directories:
        frequency_directory_name = frequency_directory.name
        dataset_names.append(f"{directory_name}/{frequency_directory_name}")

# Print all the avaliable datasets in the GIFT-Eval benchmark
print("Here are the avaliable datasets in GIFT-Eval:")
for index, name in enumerate(sorted(dataset_names)):
    print(f"• Dataset #{index + 1}: {name}")

Here are the avaliable datasets in GIFT-Eval:
• Dataset #1: LOOP_SEATTLE/5T
• Dataset #2: LOOP_SEATTLE/D
• Dataset #3: LOOP_SEATTLE/H
• Dataset #4: M_DENSE/D
• Dataset #5: M_DENSE/H
• Dataset #6: SZ_TAXI/15T
• Dataset #7: SZ_TAXI/H
• Dataset #8: bitbrains_fast_storage/5T
• Dataset #9: bitbrains_fast_storage/H
• Dataset #10: bitbrains_rnd/5T
• Dataset #11: bitbrains_rnd/H
• Dataset #12: bizitobs_application
• Dataset #13: bizitobs_l2c/5T
• Dataset #14: bizitobs_l2c/H
• Dataset #15: bizitobs_service
• Dataset #16: car_parts_with_missing
• Dataset #17: covid_deaths
• Dataset #18: electricity/15T
• Dataset #19: electricity/D
• Dataset #20: electricity/H
• Dataset #21: electricity/W
• Dataset #22: ett1/15T
• Dataset #23: ett1/D
• Dataset #24: ett1/H
• Dataset #25: ett1/W
• Dataset #26: ett2/15T
• Dataset #27: ett2/D
• Dataset #28: ett2/H
• Dataset #29: ett2/W
• Dataset #30: hierarchical_sales/D
• Dataset #31: hierarchical_sales/W
• Dataset #32: hospital
• Dataset #33: jena_weather/10T
• Dat

## Loading Datasets

Salseforce provides a `Dataset` class, to load each dataset in GIFT-Eval using the GluonTS interface. `Dataset` contains the following members:
- `training_dataset`: The training dataset
- `validation_dataset`: The validation dataset
- `test_data`: The test dataset

`training_dataset` and `validation_dataset` are iterators where each element is a dictionary containing the following keys:
- `start`: the start time of the window
- `target`: the target values of the window
- `item_id`: the id of the series
- `freq`: the frequency of the series
- `past_feat_dynamic_real`: [if exists] the past feature dynamic real values of the window

`test_data` is a tuple with two dictionaries:
- The first dictionary is for the context
- The second dictionary is for the forecast horizon
  
Each dictionary contains the same keys as above

### Example: Using the M4 Monthly Dataset

Let's load and inspect the M4 Monthly dataset using Salesforce's `Dataset` class

In [ ]:
from notebooks.src.gift_eval.data import Dataset

name = "m4_monthly"  # Name of the dataset
to_univariate = False  # Whether or not to convert the data to univariate
term = "short"  # Term of the dataset

# Instantiate dataset
dataset = Dataset(name=name, term=term, to_univariate=to_univariate)

# Print dataset's metadata
print("Dataset frequency: ", dataset.freq)
print("Prediction length: ", dataset.prediction_length)
print("Number of windows in the rolling evaluation: ", dataset.windows)

: 

Get the first time series in the training set and plot it 

In [ ]:
import matplotlib.pyplot as plt

from gluonts.dataset.util import to_pandas

# Get an iterator over the training data
training_dataset_iterator = dataset.training_dataset

# Get the first time series in the training set
train_data = next(iter(training_dataset_iterator))

# Print the keys and values in the first time series
print("First time series's keys: ", train_data.keys())
print("Item id: ", train_data["item_id"])
print("Start Date: ", train_data["start"])
print("Frequency: ", train_data["freq"])
print("Last 10 target values: ", train_data["target"][-10:])

# Convert the first time series to a pandas Series
train_series = to_pandas(train_data)

# Plot the first time series
train_series.plot()
plt.title("First Training Time Series")
plt.xlabel("time")
plt.ylabel("y")
plt.show()

: 

Get the first time series in the validation set and plot it 

In [ ]:
# Get an iterator over the validation data
val_dataset_iterator = dataset.validation_dataset

# Get the first time series in the validation set
val_data = next(iter(val_dataset_iterator))

# Print the keys and values in the first time series
print("Keys in the validation data: ", val_data.keys())
print("Item id: ", val_data["item_id"])
print("Start Date: ", val_data["start"])
print("Frequency: ", val_data["freq"])
print("Last 10 target values: ", val_data["target"][-10:])

# Convert the first time series to a pandas Series
val_series = to_pandas(val_data)

# Plot the first validation time series and show where the first training time series ends
val_series.plot()

# Add a vertical axis for where the train series ends
plt.axvline(
    x=train_series.index[-1],
    color="r",
    linestyle="--",
    label="End of train series",
)
plt.legend(["Val series", "End of train series"], loc="upper left")
plt.title("First Validation Time Series")
plt.xlabel("time")
plt.ylabel("y")
plt.show()

: 

Get the first time series in the test set and plot it 

In [ ]:
# Get an iterator over the test set
test_datset_iterator = dataset.test_data

# Get the first time series in the test set
test_data = next(iter(test_datset_iterator))

# Recall test_data is a tuple containing the context and forecast horizon

# Get an iterator the test set's contexts
test_context_iterator = dataset.test_data.input

# Get the first test time series's context
context = next(iter(test_context_iterator))

# Print the context's keys and values
print("Keys in context: ", context.keys())
print("Context Item id: ", context["item_id"])
print("Context Start Date: ", context["start"])
print("Context Frequency: ", context["freq"])
print("Context Last 10 target values: ", context["target"][-10:])

# Get an iterator over the test set's forecast horizon
test_forecast_horizon_iterator = dataset.test_data.label

# Get the first test time series's forecast horizon
forecast_horizon = next(iter(test_forecast_horizon_iterator))

# Print the forecast horizon's keys and values
print("\nKeys in forecast horizon: ", forecast_horizon.keys())
print("Forecast Horizon Item id: ", forecast_horizon["item_id"])
print("Forecast Horizon Start Date: ", forecast_horizon["start"])
print("Forecast Horizon Frequency: ", forecast_horizon["freq"])
print("Forecast Horizon Last 10 target values: ", forecast_horizon["target"][-10:])

# Convert the first test time series's context into a pandas Series
context_series = to_pandas(test_data[0])

# Convert the first test time series's forecast horizon into a pandas Series
forecast_horizon_series = to_pandas(test_data[1])

# Plot the context and forecast horizon
context_series.plot(label="Context")
forecast_horizon_series.plot(label="Forecast Horizon")
plt.legend(loc="upper left")
plt.title("First Test Time Series")
plt.xlabel("time")
plt.ylabel("y")
plt.show()

: 